In [9]:
"""
=============================================================================
Human Activity Recognition (HAR) — CNN-LSTM Architecture
=============================================================================
Production-Level Deep Learning Model for Google Colab
Dataset  : UCI HAR Dataset (Smartphone Sensors — Accelerometer + Gyroscope)
Framework: PyTorch
=============================================================================

DESIGN DECISIONS & JUSTIFICATIONS
----------------------------------
1. WHY CNN-LSTM?
   • Pure CNN catches local spatial patterns but misses long-range temporal
     dependencies across a time window.
   • Pure LSTM captures temporal dynamics but is slow and struggles with
     high-dimensional raw sensor input.
   • CNN-LSTM: CNN extracts spatial features from sensor *subsequences*,
     LSTM then models temporal evolution across those subsequences.
     → Best of both worlds: local feature extraction + global temporal modeling.
   • Transformer alternative was rejected: needs more data & compute;
     CNN-LSTM is more data-efficient for ~10K samples.

2. WHY RAW INERTIAL SIGNALS (9×128) instead of the 561 engineered features?
   • CNN can *learn* better task-specific features than hand-crafted ones.
   • Raw signals preserve full temporal ordering (needed for Conv1D).
   • More representative of real production scenarios where feature
     engineering may not be available.

3. WHY SUBSEQUENCE SPLITTING (128 → 4×32)?
   • Creates a *hierarchical* representation:
       local patterns (CNN on 32-step windows)  →  global dynamics (LSTM on 4 steps)
   • Lets CNN focus on short motifs; LSTM captures how motifs evolve.

4. WHY Conv1D (not Conv2D)?
   • Input per subsequence is (32 timesteps, 9 channels).
   • Conv1D slides along the *time* axis treating channels as features —
     the natural choice for 1-D signal processing.
   • Conv2D would treat the data like an image, which is semantically wrong.

5. OVERFITTING PREVENTION (8 complementary strategies):
   ① Spatial Dropout after CNN (drops entire feature maps)
   ② Dropout before dense classifier
   ③ Batch Normalization for training stability
   ④ L2 Weight Decay via AdamW
   ⑤ Early Stopping (patience=15)
   ⑥ Label Smoothing in CrossEntropyLoss
   ⑦ Data Augmentation (jitter + scaling)
   ⑧ Cosine Annealing LR schedule + Gradient Clipping
=============================================================================
"""

# ── Section 1: Imports ──────────────────────────────────────────────────────
import os, zipfile, urllib.request, warnings, random, time
from dataclasses import dataclass, field
from typing import Tuple, List, Optional, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)

warnings.filterwarnings("ignore")

# ── Section 2: Configuration ────────────────────────────────────────────────
@dataclass
class Config:
    """Central config — single source of truth for every hyper-parameter."""
    # ── Data ──
    data_url: str = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )
    data_dir: str = "./UCI_HAR_Dataset"
    n_channels: int   = 9       # 3 total_acc + 3 body_acc + 3 body_gyro
    window_size: int  = 128     # 2.56 s × 50 Hz
    n_classes: int    = 6       # WALKING … LAYING

    # ── Subsequence splitting ──
    n_subsequences: int  = 4    # 128/4 = 32 steps per subsequence

    # ── Model ──
    cnn_filters: List[int]  = field(default_factory=lambda: [64, 128, 128])
    cnn_kernels: List[int]  = field(default_factory=lambda: [5, 3, 3])
    lstm_hidden: int   = 128
    lstm_layers: int   = 2
    bidirectional: bool = True
    classifier_hidden: int = 128

    # ── Regularisation ──
    cnn_dropout: float    = 0.1   # spatial dropout after CNN blocks
    lstm_dropout: float   = 0.2   # between LSTM layers
    cls_dropout: float    = 0.4   # before final dense
    weight_decay: float   = 1e-4  # L2 via AdamW
    label_smoothing: float = 0.1
    grad_clip: float      = 1.0

    # ── Training ──
    epochs: int       = 10
    batch_size: int   = 64
    lr: float         = 1e-3
    val_ratio: float  = 0.15    # fraction of train used for validation
    patience: int     = 15      # early-stopping patience
    seed: int         = 42
    use_amp: bool     = True    # mixed-precision (faster on Colab GPU)

    # ── Augmentation ──
    aug_jitter_std: float  = 0.03
    aug_scale_range: Tuple[float, float] = (0.9, 1.1)

    # ── Activity labels ──
    activity_labels: List[str] = field(default_factory=lambda: [
        "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
        "SITTING", "STANDING", "LAYING"
    ])

    @property
    def steps_per_sub(self) -> int:
        return self.window_size // self.n_subsequences  # 32

    @property
    def lstm_output_dim(self) -> int:
        return self.lstm_hidden * (2 if self.bidirectional else 1)


cfg = Config()

# ── Section 3: Reproducibility ──────────────────────────────────────────────
def seed_everything(seed: int = 42):
    """Seed all sources of randomness for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {DEVICE}")

# ── Section 4: Data Download & Loading ──────────────────────────────────────

SIGNAL_FILES = [
    "total_acc_x_{}.txt", "total_acc_y_{}.txt", "total_acc_z_{}.txt",
    "body_acc_x_{}.txt",  "body_acc_y_{}.txt",  "body_acc_z_{}.txt",
    "body_gyro_x_{}.txt", "body_gyro_y_{}.txt", "body_gyro_z_{}.txt",
]

def download_dataset(cfg: Config) -> str:
    """Download & extract UCI HAR Dataset if not present. Returns dataset root."""
    zip_path = "UCI_HAR_Dataset.zip"
    root = os.path.join(cfg.data_dir, "UCI HAR Dataset")
    if os.path.isdir(root):
        print("✓ Dataset already exists.")
        return root
    print("⏳ Downloading UCI HAR Dataset …")
    urllib.request.urlretrieve(cfg.data_url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(cfg.data_dir)
    os.remove(zip_path)
    print("✓ Dataset downloaded and extracted.")
    return root


def load_signals(root: str, split: str) -> np.ndarray:
    """Load raw inertial signals → shape (n_samples, 128, 9)."""
    path = os.path.join(root, split, "Inertial Signals")
    signals = []
    for fname_template in SIGNAL_FILES:
        fname = os.path.join(path, fname_template.format(split))
        data = np.loadtxt(fname)          # (n_samples, 128)
        signals.append(data)
    # Stack along channel axis → (n_samples, 128, 9)
    return np.stack(signals, axis=-1).astype(np.float32)


def load_labels(root: str, split: str) -> np.ndarray:
    """Load activity labels (1-indexed) → 0-indexed."""
    path = os.path.join(root, split, f"y_{split}.txt")
    return np.loadtxt(path).astype(int) - 1  # 0-indexed


def standardize(
    X_train: np.ndarray, X_test: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Per-channel z-score standardisation.
    WHY: Neural networks converge faster & more stably when inputs are
    zero-mean, unit-variance. Computed on train only to prevent data leakage.
    """
    mean = X_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, 9)
    std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_test - mean) / std, mean, std


def reshape_to_subsequences(X: np.ndarray, cfg: Config) -> np.ndarray:
    """
    (n, 128, 9) → (n, n_subsequences, steps_per_sub, 9)
    Splits each window into sub-windows so CNN processes each sub-window
    and LSTM learns the temporal progression across sub-windows.
    """
    n = X.shape[0]
    return X.reshape(n, cfg.n_subsequences, cfg.steps_per_sub, cfg.n_channels)


# ── Section 5: Data Augmentation ────────────────────────────────────────────
class SensorAugmentation:
    """
    Online data augmentation for inertial sensor data.

    WHY these augmentations?
    • Jitter  – simulates sensor noise variations across devices/placements.
    • Scaling – simulates different sensor sensitivity / body mass.
    Both are physics-motivated and proven effective for HAR (Um et al., 2017).

    WHY NOT rotation/permutation?
    • Rotation changes the coordinate frame — dangerous for orientation-
      dependent activities like WALKING_UPSTAIRS vs _DOWNSTAIRS.
    • Permuting subsequences destroys temporal order that LSTM relies on.
    """
    def __init__(self, cfg: Config):
        self.jitter_std  = cfg.aug_jitter_std
        self.scale_lo, self.scale_hi = cfg.aug_scale_range

    def __call__(self, x: np.ndarray) -> np.ndarray:
        # x shape: (n_subs, steps, channels)
        # Jitter: add Gaussian noise
        noise = np.random.normal(0, self.jitter_std, x.shape).astype(np.float32)
        x = x + noise
        # Scaling: multiply by random factor per channel
        scale = np.random.uniform(
            self.scale_lo, self.scale_hi, (1, 1, x.shape[-1])
        ).astype(np.float32)
        x = x * scale
        return x


# ── Section 6: PyTorch Dataset ──────────────────────────────────────────────
class HARDataset(Dataset):
    """Custom dataset with optional augmentation."""
    def __init__(self, X: np.ndarray, y: np.ndarray,
                 augment: Optional[SensorAugmentation] = None):
        self.X = X          # (n, n_subs, steps, channels)
        self.y = y          # (n,)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].copy()  # copy to avoid mutating original
        if self.augment is not None:
            x = self.augment(x)
        return torch.from_numpy(x), torch.tensor(self.y[idx], dtype=torch.long)


# ── Section 7: Model Architecture ──────────────────────────────────────────

class CNNBlock(nn.Module):
    """
    Single Conv1D → BatchNorm → ReLU → MaxPool block.
    BatchNorm stabilises training and acts as mild regularisation.
    """
    def __init__(self, in_ch: int, out_ch: int, kernel: int):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.pool = nn.MaxPool1d(2)

    def forward(self, x):
        # x: (batch, in_ch, time)
        return self.pool(F.relu(self.bn(self.conv(x))))


class CNNFeatureExtractor(nn.Module):
    """
    Stack of Conv1D blocks ending with Global Average Pooling.

    Architecture per subsequence:
        Conv1D(9→64, k=5) → BN → ReLU → MaxPool(2)     [32→16]
        Conv1D(64→128, k=3) → BN → ReLU → MaxPool(2)   [16→8]
        Conv1D(128→128, k=3) → BN → ReLU → MaxPool(2)  [8→4]
        GlobalAvgPool → 128-dim feature vector

    WHY 3 blocks? Sufficient receptive field (5+3+3=11 steps ≈ 0.22 s)
    to capture motion primitives while keeping the model compact.

    WHY GlobalAvgPool? Reduces each feature map to a single value,
    lowering parameter count and acting as structural regularisation.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        in_channels = cfg.n_channels  # 9
        blocks = []
        for out_ch, k in zip(cfg.cnn_filters, cfg.cnn_kernels):
            blocks.append(CNNBlock(in_channels, out_ch, k))
            in_channels = out_ch
        self.blocks  = nn.Sequential(*blocks)
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(cfg.cnn_dropout)  # spatial dropout surrogate

    def forward(self, x):
        # x: (batch, time, channels) → Conv1d wants (batch, channels, time)
        x = x.permute(0, 2, 1)
        x = self.blocks(x)         # (batch, 128, T')
        x = self.gap(x).squeeze(-1)  # (batch, 128)
        return self.dropout(x)


class CNNLSTM(nn.Module):
    """
    Full CNN-LSTM model for Human Activity Recognition.

    Data flow:
        Input (batch, 4, 32, 9)
        ──── CNN (per subsequence) ────
        Reshape → (batch×4, 32, 9)
        CNNFeatureExtractor → (batch×4, 128)
        Reshape → (batch, 4, 128)
        ──── LSTM ────
        Bidirectional LSTM (2 layers) → (batch, 4, 256)
        Take last time-step → (batch, 256)
        ──── Classifier ────
        Dropout → Dense(256→128) → ReLU → Dropout → Dense(128→6)

    WHY Bidirectional LSTM?
    • Each subsequence can be informed by *future* context as well,
      improving feature aggregation. Doubles output dim (128→256).

    WHY 2 LSTM layers?
    • Provides hierarchical temporal abstraction.
    • More layers showed no benefit on this dataset size.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.cnn = CNNFeatureExtractor(cfg)

        self.lstm = nn.LSTM(
            input_size    = cfg.cnn_filters[-1],   # 128 (CNN output dim)
            hidden_size   = cfg.lstm_hidden,        # 128
            num_layers    = cfg.lstm_layers,         # 2
            batch_first   = True,
            bidirectional = cfg.bidirectional,
            dropout       = cfg.lstm_dropout if cfg.lstm_layers > 1 else 0,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(cfg.cls_dropout),
            nn.Linear(cfg.lstm_output_dim, cfg.classifier_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.cls_dropout * 0.5),
            nn.Linear(cfg.classifier_hidden, cfg.n_classes),
        )

        self._init_weights()

    def _init_weights(self):
        """Xavier init for linear layers; orthogonal for LSTM (better for long seqs)."""
        for name, param in self.named_parameters():
            if "lstm" in name and "weight" in name:
                nn.init.orthogonal_(param)
            elif "weight" in name and param.dim() >= 2:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x):
        # x: (batch, n_subs, steps, channels)
        batch, n_subs, steps, ch = x.shape

        # ── CNN: process each subsequence ──
        x = x.reshape(batch * n_subs, steps, ch)    # (B*4, 32, 9)
        cnn_out = self.cnn(x)                         # (B*4, 128)
        cnn_out = cnn_out.reshape(batch, n_subs, -1)  # (B, 4, 128)

        # ── LSTM: temporal modelling ──
        lstm_out, _ = self.lstm(cnn_out)    # (B, 4, 256)
        # Use last time-step output (captures full context)
        last_step = lstm_out[:, -1, :]      # (B, 256)

        # ── Classifier ──
        return self.classifier(last_step)   # (B, 6)


# ── Section 8: Training Engine ──────────────────────────────────────────────

class EarlyStopping:
    """Stop training when validation loss stops improving."""
    def __init__(self, patience: int = 15, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float("inf")
        self.best_state: Optional[dict] = None

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False  # keep going
        self.counter += 1
        return self.counter >= self.patience  # True = stop

    def restore(self, model: nn.Module):
        if self.best_state:
            model.load_state_dict(self.best_state)


class Trainer:
    """Manages the full training loop with mixed-precision, logging, etc."""

    def __init__(self, model: nn.Module, cfg: Config):
        self.model     = model.to(DEVICE)
        self.cfg       = cfg
        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
        self.optimizer = AdamW(
            model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
        )
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=cfg.epochs, eta_min=1e-6)
        self.scaler    = GradScaler(enabled=cfg.use_amp and DEVICE.type == "cuda")
        self.stopper   = EarlyStopping(patience=cfg.patience)
        self.history: Dict[str, List[float]] = {
            "train_loss": [], "val_loss": [],
            "train_acc": [], "val_acc": [], "lr": []
        }

    def _run_epoch(self, loader: DataLoader, train: bool = True):
        self.model.train() if train else self.model.eval()
        total_loss, correct, total = 0.0, 0, 0

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(DEVICE, non_blocking=True)
                y_batch = y_batch.to(DEVICE, non_blocking=True)

                use_amp = self.cfg.use_amp and DEVICE.type == "cuda"
                with autocast(enabled=use_amp):
                    logits = self.model(X_batch)
                    loss   = self.criterion(logits, y_batch)

                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.cfg.grad_clip
                    )
                    self.scaler.step(self.optimizer)
                    self.scaler.update()

                total_loss += loss.item() * X_batch.size(0)
                preds       = logits.argmax(dim=1)
                correct    += (preds == y_batch).sum().item()
                total      += X_batch.size(0)

        return total_loss / total, correct / total

    def fit(self, train_loader: DataLoader, val_loader: DataLoader):
        """Full training loop with early stopping and logging."""
        print(f"\n{'='*60}")
        print(f"  Training CNN-LSTM  |  {sum(p.numel() for p in self.model.parameters()):,} parameters")
        print(f"{'='*60}\n")

        for epoch in range(1, self.cfg.epochs + 1):
            t0 = time.time()
            train_loss, train_acc = self._run_epoch(train_loader, train=True)
            val_loss,   val_acc   = self._run_epoch(val_loader,   train=False)
            self.scheduler.step()
            lr = self.scheduler.get_last_lr()[0]

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)
            self.history["lr"].append(lr)

            elapsed = time.time() - t0
            print(
                f"Epoch {epoch:3d}/{self.cfg.epochs} │ "
                f"Train Loss {train_loss:.4f}  Acc {train_acc:.4f} │ "
                f"Val Loss {val_loss:.4f}  Acc {val_acc:.4f} │ "
                f"LR {lr:.2e} │ {elapsed:.1f}s"
            )

            if self.stopper.step(val_loss, self.model):
                print(f"\n⚠ Early stopping at epoch {epoch}. "
                      f"Best val loss: {self.stopper.best_loss:.4f}")
                break

        self.stopper.restore(self.model)
        print("✓ Restored best model weights.\n")
        return self.history


# ── Section 9: Evaluation & Visualisation ───────────────────────────────────

class Evaluator:
    """Generate all evaluation metrics and plots."""

    def __init__(self, model: nn.Module, cfg: Config):
        self.model = model.to(DEVICE)
        self.cfg   = cfg

    @torch.no_grad()
    def predict(self, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
        self.model.eval()
        all_preds, all_labels = [], []
        for X, y in loader:
            X = X.to(DEVICE, non_blocking=True)
            preds = self.model(X).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(y.numpy())
        return np.concatenate(all_preds), np.concatenate(all_labels)

    def full_report(self, loader: DataLoader, history: dict, save_dir: str = "."):
        preds, labels = self.predict(loader)
        acc = accuracy_score(labels, preds)
        f1  = f1_score(labels, preds, average="weighted")
        print(f"\n{'='*60}")
        print(f"  TEST RESULTS  —  Accuracy: {acc:.4f}  |  F1 (weighted): {f1:.4f}")
        print(f"{'='*60}\n")
        print(classification_report(
            labels, preds, target_names=self.cfg.activity_labels, digits=4
        ))

        # ── Plot 1: Training curves ──
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(history["train_loss"], label="Train")
        axes[0].plot(history["val_loss"],   label="Val")
        axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
        axes[1].plot(history["train_acc"], label="Train")
        axes[1].plot(history["val_acc"],   label="Val")
        axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
        axes[2].plot(history["lr"])
        axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Epoch")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150)
        plt.show()

        # ── Plot 2: Confusion matrix ──
        cm = confusion_matrix(labels, preds)
        fig, ax = plt.subplots(figsize=(8, 7))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=self.cfg.activity_labels,
                    yticklabels=self.cfg.activity_labels, ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix — Accuracy {acc:.4f}")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "confusion_matrix.png"), dpi=150)
        plt.show()

        # ── Plot 3: Per-class accuracy ──
        per_class = cm.diagonal() / cm.sum(axis=1)
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(self.cfg.activity_labels, per_class, color=sns.color_palette("viridis", 6))
        for bar, v in zip(bars, per_class):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.2%}", ha="center", fontsize=10)
        ax.set_ylim(0, 1.1); ax.set_ylabel("Accuracy"); ax.set_title("Per-Class Accuracy")
        plt.xticks(rotation=30, ha="right"); plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "per_class_accuracy.png"), dpi=150)
        plt.show()

        return {"accuracy": acc, "f1_weighted": f1, "per_class": per_class}
# ── Section: Model Info & Architecture Printer ──────────────────────────────

def print_model_architecture(model: nn.Module, cfg: Config, sample_input: torch.Tensor = None):
    """
    Print comprehensive model architecture information including:
    - ASCII architecture diagram
    - Layer-by-layer breakdown with shapes and parameters
    - Parameter statistics
    - Memory footprint estimate
    - Design rationale summary
    """

    # ═══════════════════════════════════════════════════════════════════════
    # HEADER
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "╔" + "═" * 78 + "╗")
    print("║" + "  CNN-LSTM MODEL ARCHITECTURE — HUMAN ACTIVITY RECOGNITION".center(78) + "║")
    print("╚" + "═" * 78 + "╝")

    # ═══════════════════════════════════════════════════════════════════════
    # ASCII ARCHITECTURE DIAGRAM
    # ═══════════════════════════════════════════════════════════════════════
    print("""
┌─────────────────────────────────────────────────────────────────────────────────┐
│                         ARCHITECTURE DATA FLOW                                  │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│   INPUT: Raw Sensor Signals                                                     │
│   ┌─────────────────────────────────────┐                                      │
│   │  (Batch, 4 subseqs, 32 steps, 9 ch) │  ← 9 channels: 3 total_acc +         │
│   └──────────────────┬──────────────────┘    3 body_acc + 3 body_gyro          │
│                      │                                                      │
│                      ▼                                                      │
│   RESHAPE: (Batch×4, 32, 9)  ← Flatten subsequences for batch processing     │
│                      │                                                      │
│                      ▼                                                      │
│   ┌──────────────────────────────────────────────────────────────────┐       │
│   │                    CNN FEATURE EXTRACTOR                          │       │
│   │  ┌─────────────────────────────────────────────────────────────┐ │       │
│   │  │ Block 1: Conv1d(9→64, k=5) → BN → ReLU → MaxPool(2)       │ │       │
│   │  │          (B×4, 32, 9)  →  (B×4, 64, 16)                   │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Block 2: Conv1d(64→128, k=3) → BN → ReLU → MaxPool(2)     │ │       │
│   │  │          (B×4, 64, 16)  →  (B×4, 128, 8)                  │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Block 3: Conv1d(128→128, k=3) → BN → ReLU → MaxPool(2)    │ │       │
│   │  │          (B×4, 128, 8)  →  (B×4, 128, 4)                  │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ GAP: AdaptiveAvgPool1d(1) → squeeze                       │ │       │
│   │  │          (B×4, 128, 4)  →  (B×4, 128)                     │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Dropout(p=0.1)                                             │ │       │
│   │  └─────────────────────────────────────────────────────────────┘ │       │
│   └──────────────────────────┬───────────────────────────────────────┘       │
│                              │                                               │
│                              ▼                                               │
│   RESHAPE: (Batch, 4, 128)  ← Reassemble temporal subsequences               │
│                              │                                               │
│                              ▼                                               │
│   ┌──────────────────────────────────────────────────────────────────┐       │
│   │                    BIDIRECTIONAL LSTM                             │       │
│   │  ┌─────────────────────────────────────────────────────────────┐ │       │
│   │  │ Layer 1: LSTM(128→128, bidir=True, dropout=0.2)            │ │       │
│   │  │          (Batch, 4, 128) → (Batch, 4, 256)                 │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Layer 2: LSTM(256→128, bidir=True)                         │ │       │
│   │  │          (Batch, 4, 256) → (Batch, 4, 256)                 │ │       │
│   │  └─────────────────────────────────────────────────────────────┘ │       │
│   └──────────────────────────┬───────────────────────────────────────┘       │
│                              │                                               │
│                              ▼                                               │
│   TAKE LAST TIMESTEP: (Batch, 256)                                            │
│                              │                                               │
│                              ▼                                               │
│   ┌──────────────────────────────────────────────────────────────────┐       │
│   │                    CLASSIFIER HEAD                                │       │
│   │  ┌─────────────────────────────────────────────────────────────┐ │       │
│   │  │ Dropout(p=0.4)                                             │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Linear(256 → 128)  +  ReLU                                 │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Dropout(p=0.2)                                             │ │       │
│   │  ├─────────────────────────────────────────────────────────────┤ │       │
│   │  │ Linear(128 → 6)  →  Logits                                │ │       │
│   │  └─────────────────────────────────────────────────────────────┘ │       │
│   └──────────────────────────┬───────────────────────────────────────┘       │
│                              │                                               │
│                              ▼                                               │
│   OUTPUT: (Batch, 6)  →  Activity Probabilities (Softmax)                    │
│                                                                                 │
│   Classes: WALKING | WALKING_UPSTAIRS | WALKING_DOWNSTAIRS |                   │
│            SITTING | STANDING | LAYING                                         │
└─────────────────────────────────────────────────────────────────────────────────┘
""")

    # ═══════════════════════════════════════════════════════════════════════
    # LAYER-BY-LAYER TABLE (like Keras model.summary())
    # ═══════════════════════════════════════════════════════════════════════
    print("┌─────────────────────────────────────────────────────────────────────────────────┐")
    print("║                        LAYER-BY-LAYER SUMMARY                                    ║")
    print("╠═══════════════════════════════╦═════════════════════╦═══════════════════════════╣")
    print("║ Layer (type)                  ║ Output Shape        ║ Param #                   ║")
    print("╠═══════════════════════════════╬═════════════════════╬═══════════════════════════╣")

    batch = cfg.batch_size
    rows = []

    # Input
    rows.append(("Input", f"({batch}, 4, 32, 9)", 0))

    # Reshape
    rows.append(("Reshape", f"({batch}×4, 32, 9)", 0))

    # CNN blocks
    shape_ch, shape_t = 9, 32
    for i, (out_ch, k) in enumerate(zip(cfg.cnn_filters, cfg.cnn_kernels)):
        # Conv1d params: in_ch * out_ch * kernel + out_ch (bias)
        conv_params = shape_ch * out_ch * k + out_ch
        # BN params: 2 * out_ch (weight + bias) + 2 * out_ch (running_mean + var, not learned but stored)
        bn_learned = 2 * out_ch
        shape_t = shape_t // 2  # MaxPool halves time
        block_params = conv_params + bn_learned
        rows.append((
            f"  Conv1d-BN-ReLU-Pool [{i+1}]",
            f"(B×4, {out_ch}, {shape_t})",
            block_params
        ))
        shape_ch = out_ch

    # GAP
    gap_params = 0
    rows.append(("  GlobalAvgPool1d", f"(B×4, {shape_ch})", gap_params))

    # CNN Dropout
    rows.append(("  Dropout (cnn)", f"(B×4, {shape_ch})", 0))

    # Reshape to LSTM input
    rows.append(("Reshape", f"({batch}, 4, {shape_ch})", 0))

    # LSTM layers
    for i in range(cfg.lstm_layers):
        if i == 0:
            in_dim = shape_ch
        else:
            in_dim = cfg.lstm_hidden * 2  # bidirectional doubles it
        # LSTM params: 4 * [(in_dim + hidden + 1) * hidden] * directions
        lstm_params = 4 * (in_dim + cfg.lstm_hidden + 1) * cfg.lstm_hidden
        if cfg.bidirectional:
            lstm_params *= 2
        rows.append((
            f"  BiLSTM [{i+1}]",
            f"({batch}, 4, {cfg.lstm_hidden * 2})",
            lstm_params
        ))

    # Take last timestep
    rows.append(("Take Last Timestep", f"({batch}, {cfg.lstm_output_dim})", 0))

    # Classifier
    # Dropout
    rows.append(("  Dropout (cls)", f"({batch}, {cfg.lstm_output_dim})", 0))
    # Linear 1
    fc1_params = cfg.lstm_output_dim * cfg.classifier_hidden + cfg.classifier_hidden
    rows.append((
        f"  Linear ({cfg.lstm_output_dim}→{cfg.classifier_hidden}) + ReLU",
        f"({batch}, {cfg.classifier_hidden})",
        fc1_params
    ))
    # Dropout
    rows.append(("  Dropout (cls)", f"({batch}, {cfg.classifier_hidden})", 0))
    # Linear 2
    fc2_params = cfg.classifier_hidden * cfg.n_classes + cfg.n_classes
    rows.append((
        f"  Linear ({cfg.classifier_hidden}→{cfg.n_classes})",
        f"({batch}, {cfg.n_classes})",
        fc2_params
    ))

    total_params = sum(r[2] for r in rows)

    for name, shape, params in rows:
        # Truncate name if too long
        name_display = name[:29].ljust(29)
        shape_display = shape[:19].ljust(19)
        params_display = f"{params:,}".rjust(25)
        print(f"║ {name_display} ║ {shape_display} ║ {params_display} ║")

    print("╠═══════════════════════════════╬═════════════════════╬═══════════════════════════╣")
    total_display = f"Total: {total_params:,}".rjust(25)
    print(f"║ {'':29} ║ {'':19} ║ {total_display} ║")
    print("╚═══════════════════════════════╩═════════════════════╩═══════════════════════════╝")

    # ═══════════════════════════════════════════════════════════════════════
    # PARAMETER BREAKDOWN BY COMPONENT
    # ═══════════════════════════════════════════════════════════════════════
    cnn_params  = sum(p.numel() for p in model.cnn.parameters())
    lstm_params = sum(p.numel() for p in model.lstm.parameters())
    cls_params  = sum(p.numel() for p in model.classifier.parameters())
    trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen      = total_params - trainable

    print(f"""
┌─────────────────────────────────────────────────────────────┐
│                  PARAMETER BREAKDOWN                        │
├─────────────────────────────┬───────────────┬───────────────┤
│ Component                   │   Params      │    % of Total │
├─────────────────────────────┼───────────────┼───────────────┤
│ CNN Feature Extractor       │ {cnn_params:>11,} │ {100*cnn_params/total_params:>11.2f}% │
│ Bidirectional LSTM          │ {lstm_params:>11,} │ {100*lstm_params/total_params:>11.2f}% │
│ Classifier Head             │ {cls_params:>11,} │ {100*cls_params/total_params:>11.2f}% │
├─────────────────────────────┼───────────────┼───────────────┤
│ Trainable Parameters        │ {trainable:>11,} │ {100*trainable/total_params:>11.2f}% │
│ Frozen Parameters           │ {frozen:>11,} │ {100*frozen/total_params:>11.2f}% │
├─────────────────────────────┼───────────────┼───────────────┤
│ TOTAL                       │ {total_params:>11,} │       100.00% │
└─────────────────────────────┴───────────────┴───────────────┘
""")

    # ═══════════════════════════════════════════════════════════════════════
    # MEMORY FOOTPRINT ESTIMATE
    # ═══════════════════════════════════════════════════════════════════════
    # FP32 = 4 bytes per param; AdamW stores 2 momentum buffers = 3x total
    param_bytes     = total_params * 4
    optimizer_bytes = total_params * 4 * 2  # first_moment + second_moment
    total_memory    = param_bytes + optimizer_bytes

    print(f"""┌─────────────────────────────────────────────────────────────┐
│                  MEMORY FOOTPRINT (FP32)                    │
├─────────────────────────────┬───────────────┬───────────────┤
│ Item                        │   Size        │    Approx.    │
├─────────────────────────────┼───────────────┼───────────────┤
│ Model weights (FP32)        │ {param_bytes:>11,} B │ {param_bytes/1024/1024:>9.2f} MB │
│ AdamW optimizer state       │ {optimizer_bytes:>11,} B │ {optimizer_bytes/1024/1024:>9.2f} MB │
├─────────────────────────────┼───────────────┼───────────────┤
│ Total (params + optimizer)  │ {total_memory:>11,} B │ {total_memory/1024/1024:>9.2f} MB │
└─────────────────────────────┴───────────────┴───────────────┘""")

    # AMP halves memory for activations
    if cfg.use_amp:
        print(f"  ℹ  With AMP (FP16), activation memory is ~halved during training.")

    # ═══════════════════════════════════════════════════════════════════════
    # COMPUTATIONAL COMPLEXITY (FLOPS)
    # ═══════════════════════════════════════════════════════════════════════
    # Approximate FLOPs for a single forward pass
    flops = 0
    # CNN
    t = 32
    in_ch = 9
    for out_ch, k in zip(cfg.cnn_filters, cfg.cnn_kernels):
        flops += batch * 4 * out_ch * in_ch * k * t  # 4 subsequences
        t //= 2
        in_ch = out_ch
    # LSTM (approximate: 4 * hidden * (input + hidden) per direction per layer per timestep)
    for i in range(cfg.lstm_layers):
        inp = cfg.cnn_filters[-1] if i == 0 else cfg.lstm_hidden * 2
        flops += batch * 4 * 4 * cfg.lstm_hidden * (inp + cfg.lstm_hidden) * cfg.n_subsequences
    # Classifier
    flops += batch * (cfg.lstm_output_dim * cfg.classifier_hidden + cfg.classifier_hidden * cfg.n_classes)

    print(f"""
┌─────────────────────────────────────────────────────────────┐
│             APPROXIMATE COMPUTE (per forward pass)          │
├─────────────────────────────┬───────────────────────────────┤
│ FLOPs (batch={batch})          │ {flops:>25,} │
│ MFLOPs                      │ {flops/1e6:>25.2f} │
│ GFLOPs                      │ {flops/1e9:>25.4f} │
└─────────────────────────────┴───────────────────────────────┘
""")

    # ═══════════════════════════════════════════════════════════════════════
    # DESIGN DECISIONS SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    print("""┌─────────────────────────────────────────────────────────────────────────────────┐
│                         DESIGN DECISIONS SUMMARY                                   │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│  ARCHITECTURE                                                                   │
│  ├─ Why CNN-LSTM hybrid?                                                        │
│  │   CNN extracts local spatial features from short windows;                    │
│  │   LSTM captures long-range temporal dependencies across windows.             │
│  │   → Outperforms pure CNN (misses temporal) and pure LSTM (slow, noisy).     │
│  │                                                                               │
│  ├─ Why subsequence splitting (128 → 4×32)?                                     │
│  │   Creates hierarchical representation: local motifs → global dynamics.       │
│  │   CNN focuses on 32-step patterns; LSTM on 4-step evolution.                 │
│  │                                                                               │
│  ├─ Why Conv1d (not Conv2d)?                                                    │
│  │   Signals are 1-D time series. Conv2d treats data like images,               │
│  │   which is semantically incorrect and adds unnecessary parameters.           │
│  │                                                                               │
│  ├─ Why Bidirectional LSTM?                                                     │
│  │   Each subsequence can leverage future context for better feature            │
│  │   aggregation. Doubles output dim (128→256) but improves accuracy.           │
│  │                                                                               │
│  └─ Why Global Average Pooling (not Flatten)?                                   │
│      Reduces parameters drastically and provides spatial invariance.            │
│                                                                                 │
│  INPUT                                                                          │
│  ├─ Why raw 9-channel signals (not 561 engineered features)?                    │
│  │   CNN learns task-specific features better than hand-crafted ones.           │
│  │   Raw signals preserve temporal ordering needed for Conv1d.                  │
│  │   More representative of production scenarios without feature engineering.   │
│  │                                                                               │
│  └─ Why per-channel z-score standardisation (train only)?                       │
│      Zero-mean, unit-variance inputs accelerate convergence.                    │
│      Computed on train only to prevent data leakage to test set.                │
│                                                                                 │
│  REGULARISATION (8 strategies)                                                  │
│  ├─ ① Spatial Dropout after CNN (drops entire feature maps)                     │
│  ├─ ② Dropout before dense classifier (p=0.4, 0.2)                             │
│  ├─ ③ Batch Normalisation (training stability + mild regularisation)           │
│  ├─ ④ L2 Weight Decay via AdamW (λ=1e-4)                                       │
│  ├─ ⑤ Early Stopping (patience=15)                                              │
│  ├─ ⑥ Label Smoothing in CrossEntropy (ε=0.1, prevents overconfident preds)    │
│  ├─ ⑦ Data Augmentation (jitter σ=0.03 + scaling [0.9, 1.1])                   │
│  └─ ⑧ Cosine Annealing LR + Gradient Clipping (max_norm=1.0)                   │
│                                                                                 │
│  TRAINING                                                                       │
│  ├─ Optimiser: AdamW (decoupled weight decay, better generalisation)           │
│  ├─ LR Schedule: Cosine Annealing (warm restarts not needed for 100 epochs)    │
│  ├─ Mixed Precision: FP16 activations (2× speed on GPU, minimal accuracy loss) │
│  └─ Gradient Clipping: Prevents exploding gradients in LSTM                   │
│                                                                                 │
│  WHY NOT TRANSFORMER?                                                           │
│  └─ Self-attention needs more data and compute (~10K samples is too small).    │
│     CNN-LSTM is more data-efficient and achieves >93% accuracy on UCI HAR.     │
│                                                                                 │
└─────────────────────────────────────────────────────────────────────────────────┘
""")

    # ═══════════════════════════════════════════════════════════════════════
    # DETAILED LAYER INFORMATION
    # ═══════════════════════════════════════════════════════════════════════
    print("┌─────────────────────────────────────────────────────────────────────────────────┐")
    print("║                    DETAILED MODULE INFORMATION                                  ║")
    print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    def count_params(module):
        return sum(p.numel() for p in module.parameters())

    def format_size(module):
        return f"{count_params(module) * 4 / 1024:.1f} KB"

    # CNN Blocks
    for i, block in enumerate(model.cnn.blocks):
        conv = block.conv
        bn = block.bn
        print(f"║  CNN Block {i+1}:                                                                    ║")
        print(f"║    Conv1d: in={conv.in_channels}, out={conv.out_channels}, kernel={conv.kernel_size[0]}, "
              f"padding={conv.padding[0]}, stride={conv.stride[0]}{'':>18}║")
        print(f"║      Parameters: {count_params(conv):>8,}  │  Size: {format_size(conv):>10}{'':>16}║")
        print(f"║    BatchNorm1d: features={bn.num_features}{'':>42}║")
        print(f"║      Parameters: {count_params(bn):>8,}  │  Size: {format_size(bn):>10}{'':>16}║")
        print(f"║    MaxPool1d: kernel_size={block.pool.kernel_size}{'':>39}║")
        print(f"║    Activation: ReLU{'':>52}║")
        print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    # GAP
    print(f"║  GlobalAvgPool1d: output_size=1{'':>44}║")
    print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    # Dropout
    print(f"║  Dropout (CNN): p={model.cnn.dropout.p}{'':>48}║")
    print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    # LSTM
    lstm = model.lstm
    print(f"║  LSTM:                                                                          ║")
    print(f"║    input_size={lstm.input_size}, hidden_size={lstm.hidden_size}, "
          f"num_layers={lstm.num_layers}{'':>8}║")
    print(f"║    bidirectional={lstm.bidirectional}, dropout={lstm.dropout}, "
          f"batch_first={lstm.batch_first}{'':>7}║")
    print(f"║      Parameters: {count_params(lstm):>8,}  │  Size: {format_size(lstm):>10}{'':>16}║")
    print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    # Classifier layers
    for i, layer in enumerate(model.classifier):
        if isinstance(layer, nn.Linear):
            print(f"║  Linear[{i}]: in={layer.in_features}, out={layer.out_features}, "
                  f"bias={layer.bias is not None}{'':>10}║")
            print(f"║    Parameters: {count_params(layer):>8,}  │  Size: {format_size(layer):>10}{'':>16}║")
        elif isinstance(layer, nn.Dropout):
            print(f"║  Dropout[{i}]: p={layer.p}{'':>52}║")
        elif isinstance(layer, nn.ReLU):
            print(f"║  ReLU[{i}]{'':>60}║")
        print("╠═════════════════════════════════════════════════════════════════════════════════╣")

    print(f"║  Weight Initialisation:                                                          ║")
    print(f"║    LSTM weights → Orthogonal (better for long sequences)                        ║")
    print(f"║    Linear weights → Xavier Uniform                                                ║")
    print(f"║    Biases → Zeros                                                                 ║")
    print("╚═════════════════════════════════════════════════════════════════════════════════╝")

    # ═══════════════════════════════════════════════════════════════════════
    # HYPERPARAMETER SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    print(f"""
┌─────────────────────────────────────────────────────────────┐
│                  HYPERPARAMETER CONFIGURATION                │
├─────────────────────────────┬───────────────────────────────┤
│ Data                        │                               │
│   Window size               │ {str(cfg.window_size):>29} │
│   Channels                  │ {str(cfg.n_channels):>29} │
│   Subsequences              │ {str(cfg.n_subsequences):>29} │
│   Steps per subsequence     │ {str(cfg.steps_per_sub):>29} │
│   Classes                   │ {str(cfg.n_classes):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ CNN                         │                               │
│   Filters                   │ {str(cfg.cnn_filters):>29} │
│   Kernels                   │ {str(cfg.cnn_kernels):>29} │
│   Dropout                   │ {str(cfg.cnn_dropout):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ LSTM                        │                               │
│   Hidden size               │ {str(cfg.lstm_hidden):>29} │
│   Layers                    │ {str(cfg.lstm_layers):>29} │
│   Bidirectional             │ {str(cfg.bidirectional):>29} │
│   Dropout                   │ {str(cfg.lstm_dropout):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ Classifier                  │                               │
│   Hidden dim                │ {str(cfg.classifier_hidden):>29} │
│   Dropout                   │ {str(cfg.cls_dropout):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ Training                    │                               │
│   Epochs                    │ {str(cfg.epochs):>29} │
│   Batch size                │ {str(cfg.batch_size):>29} │
│   Learning rate             │ {str(cfg.lr):>29} │
│   Weight decay              │ {str(cfg.weight_decay):>29} │
│   Label smoothing           │ {str(cfg.label_smoothing):>29} │
│   Gradient clip             │ {str(cfg.grad_clip):>29} │
│   Early stopping patience   │ {str(cfg.patience):>29} │
│   Mixed precision (AMP)     │ {str(cfg.use_amp):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ Augmentation                │                               │
│   Jitter std                │ {str(cfg.aug_jitter_std):>29} │
│   Scale range               │ {str(cfg.aug_scale_range):>29} │
├─────────────────────────────┼───────────────────────────────┤
│ Validation split            │ {str(cfg.val_ratio):>29} │
│ Random seed                 │ {str(cfg.seed):>29} │
│ Device                     │ {str(DEVICE):>29} │
└─────────────────────────────┴───────────────────────────────┘
""")

    # ═══════════════════════════════════════════════════════════════════════
    # ACTUAL PYTORCH STATE DICT (for verification)
    # ═══════════════════════════════════════════════════════════════════════
    print("┌─────────────────────────────────────────────────────────────────────────────────┐")
    print("║               STATE DICT VERIFICATION (all parameters)                          ║")
    print("╠═════════════════════════════════════════════════════╦═══════════════════════════╣")
    print("║ Parameter Name                                       ║ Shape                     ║")
    print("╠═════════════════════════════════════════════════════╬═══════════════════════════╣")

    for name, param in model.named_parameters():
        shape_str = str(tuple(param.shape))
        name_display = name[:52].ljust(52)
        shape_display = shape_str.rjust(25)
        print(f"║ {name_display} ║ {shape_display} ║")

    print("╚═════════════════════════════════════════════════════╩═══════════════════════════╝")

    # ═══════════════════════════════════════════════════════════════════════
    # FORWARD PASS SHAPE VERIFICATION
    # ═══════════════════════════════════════════════════════════════════════
    print("\n┌─────────────────────────────────────────────────────────────────────────────────┐")
    print("║               FORWARD PASS SHAPE VERIFICATION                                     ║")
    print("╚═════════════════════════════════════════════════════════════════════════════════╝")

    if sample_input is None:
        sample_input = torch.randn(2, cfg.n_subsequences, cfg.steps_per_sub, cfg.n_channels)

    model.eval()
    with torch.no_grad():
        x = sample_input
        print(f"\n  Input shape:                {str(tuple(x.shape))}")

        # After CNN reshape
        batch_sz, n_subs, steps, ch = x.shape
        x_reshaped = x.reshape(batch_sz * n_subs, steps, ch)
        print(f"  After CNN reshape:         {str(tuple(x_reshaped.shape))}")

        # Through CNN
        cnn_out = model.cnn(x_reshaped)
        print(f"  After CNN + GAP + Dropout: {str(tuple(cnn_out.shape))}")

        # After LSTM reshape
        x_lstm = cnn_out.reshape(batch_sz, n_subs, -1)
        print(f"  After LSTM reshape:        {str(tuple(x_lstm.shape))}")

        # Through LSTM
        lstm_out, _ = model.lstm(x_lstm)
        print(f"  After BiLSTM:              {str(tuple(lstm_out.shape))}")

        # Take last timestep
        last = lstm_out[:, -1, :]
        print(f"  After take last timestep:  {str(tuple(last.shape))}")

        # Through classifier
        logits = model.classifier(last)
        print(f"  Final output (logits):     {str(tuple(logits.shape))}")

        # Softmax
        probs = F.softmax(logits, dim=1)
        print(f"  Final output (probs):      {str(tuple(probs.shape))}")
        print(f"  Prob sum per sample:       {probs.sum(dim=1).tolist()}")

    print(f"\n  ✓ All shapes verified successfully!\n")
    print("╔" + "═" * 78 + "╗")
    print("║" + "  END OF MODEL ARCHITECTURE REPORT".center(78) + "║")
    print("╚" + "═" * 78 + "╝\n")




# ── Section 10: Main Pipeline ──────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  CNN-LSTM  —  Human Activity Recognition")
    print("=" * 60)

    # ── 1. Data ──
    root = download_dataset(cfg)
    X_train_raw = load_signals(root, "train")  # (7352, 128, 9)
    y_train_raw = load_labels(root, "train")    # (7352,)
    X_test_raw  = load_signals(root, "test")    # (2947, 128, 9)
    y_test_raw  = load_labels(root, "test")     # (2947,)
    print(f"✓ Train: {X_train_raw.shape}  Test: {X_test_raw.shape}")

    # ── 2. Standardise ──
    X_train_std, X_test_std, mean, std = standardize(X_train_raw, X_test_raw)

    # ── 3. Reshape into subsequences ──
    X_train_sub = reshape_to_subsequences(X_train_std, cfg)  # (7352, 4, 32, 9)
    X_test_sub  = reshape_to_subsequences(X_test_std, cfg)

    # ── 4. Train / Val split ──
    n_val   = int(len(y_train_raw) * cfg.val_ratio)
    n_train = len(y_train_raw) - n_val
    indices = np.random.permutation(len(y_train_raw))
    train_idx, val_idx = indices[:n_train], indices[n_train:]

    augmentor = SensorAugmentation(cfg)
    train_ds = HARDataset(X_train_sub[train_idx], y_train_raw[train_idx], augment=augmentor)
    val_ds   = HARDataset(X_train_sub[val_idx],   y_train_raw[val_idx],   augment=None)
    test_ds  = HARDataset(X_test_sub,              y_test_raw,             augment=None)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"✓ Loaders — Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

    # ── 5. Model ──
    model = CNNLSTM(cfg)
    print_model_architecture(model, cfg)
    print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # ── 6. Train ──
    trainer = Trainer(model, cfg)
    history = trainer.fit(train_loader, val_loader)

    # ── 7. Evaluate on held-out test set ──
    evaluator = Evaluator(model, cfg)
    results   = evaluator.full_report(test_loader, history)

    # ── 8. Save model ──
    save_path = "har_cnn_lstm_model.pth"
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": cfg,
        "standardization": {"mean": mean, "std": std},
        "results": results,
    }, save_path)
    print(f"\n✓ Model saved to {save_path}")
    print(f"  Final Test Accuracy : {results['accuracy']:.4f}")
    print(f"  Final Test F1       : {results['f1_weighted']:.4f}")

if __name__ == "__main__":
    main()

✓ Device: cuda
  CNN-LSTM  —  Human Activity Recognition
✓ Dataset already exists.
✓ Train: (7352, 128, 9)  Test: (2947, 128, 9)
✓ Loaders — Train: 6250  Val: 1102  Test: 2947

╔══════════════════════════════════════════════════════════════════════════════╗
║            CNN-LSTM MODEL ARCHITECTURE — HUMAN ACTIVITY RECOGNITION          ║
╚══════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────────┐
│                         ARCHITECTURE DATA FLOW                                  │
├─────────────────────────────────────────────────────────────────────────────────┤
│                                                                                 │
│   INPUT: Raw Sensor Signals                                                     │
│   ┌─────────────────────────────────────┐                                      │
│   │  (Batch, 4 subseqs, 32 steps, 9 ch) │  ← 9 channels: 3 total_acc +    